# EcoIzge · Colab eğitim şablonu

Kopyala: `notebooks/colab/<gorev>-<ad>.ipynb`. Kurallar: `.claude/skills/colab-train/SKILL.md`.

1. `GIT_SHA` ve `GOREV` değerlerini doldur.
2. Çalışma zamanını yeniden başlat ve **tüm hücreleri baştan sona** çalıştır.
3. Son hücrenin yazdığı `run.json`'u repoya `runs/<run_id>/run.json` olarak ekle.


In [ ]:
# 1 · Parametreler — yalnızca burayı düzenle
GOREV = "S14"  # tasks/ içindeki görev
GIT_SHA = "DOLDUR"  # eğitilecek commit (git log --oneline)
VERI_SURUMU = "reviewed-YYYY-AA-GG"  # Drive'daki etiket klasörü adı
KATALOG = "catalog_v0"
TOHUM = 43
ALTIN_KUME = False  # yalnızca evaluator'ın nihai deneyinde True

In [ ]:
# 2 · Repo (belirli commit) ve bağımlılıklar
!git clone -q https://github.com/MevoTech/ecoizge.git /content/ecoizge
%cd /content/ecoizge
!git checkout -q $GIT_SHA
!pip install -q -r requirements-train.txt

In [ ]:
# 3 · Drive: veri okunur, model yazılır (model dosyası git'e girmez)
from google.colab import drive

drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/ecoizge"
DATA = f"{DRIVE}/data/{VERI_SURUMU}"
MODELS = f"{DRIVE}/models"

In [ ]:
# 4 · Tohum (tekrarlanabilirlik)
import os
import random

import numpy as np

random.seed(TOHUM)
np.random.seed(TOHUM)
os.environ["PYTHONHASHSEED"] = str(TOHUM)
try:
    import torch

    torch.manual_seed(TOHUM)
    torch.cuda.manual_seed_all(TOHUM)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
except ImportError:
    pass
assert not ALTIN_KUME or GOREV in {"S22", "S23"}, "Altın küme yalnızca nihai deneylerde"

In [ ]:
# 5 · Eğitim / değerlendirme — örnek: taban çizgisi (gap-baseline)
import glob
import json
import subprocess

labels = sorted(glob.glob(f"{DATA}/*.csv"))
subprocess.run(
    [
        "python",
        ".claude/skills/gap-baseline/scripts/baseline.py",
        "--labels",
        *labels,
        "--catalog",
        f"data/catalog/{KATALOG}.csv",
        "--seed",
        str(TOHUM),
        "--out",
        "/content/metrikler.json",
    ],
    check=True,
)
with open("/content/metrikler.json", encoding="utf-8") as fh:
    metrikler = json.load(fh)
print({k: metrikler[k] for k in ("f1_makro", "f1_karsilanan", "kat_sayisi")})

In [ ]:
# 6 · Koşu kaydı (run.json) — durum: onay_bekliyor
import json
import subprocess

ozet = {k: metrikler[k] for k in ("f1_makro", "f1_karsilanan", "kat_sayisi", "cift_sayisi")}
cmd = [
    "python",
    ".claude/skills/colab-train/scripts/run_record.py",
    "create",
    "--gorev",
    GOREV,
    "--veri-surumu",
    VERI_SURUMU,
    "--katalog-surumu",
    KATALOG,
    "--tohum",
    str(TOHUM),
    "--metrikler",
    json.dumps(ozet),
    "--out",
    f"{DRIVE}/runs",
]
if ALTIN_KUME:
    cmd.append("--altin-kume")
subprocess.run(cmd, check=True)